#### Это ноутбук для запуска разметки моделью qwen специально для kaggle, чтобы использовать gpu.
Для запуска другой модели нужно заменить поле qwen в ячейках.

In [ ]:
from dataclasses import dataclass
import json
from math import isclose
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.patches import Rectangle

RESULTS = Path("results")
@dataclass(frozen=True)
class Config:
    name: str
    backend: str
    model_id: str
    prompt: str | tuple[str, ...]
    revision: str = "main"
    candidate_threshold: float = 0.0
    threshold: float = 0.0
    text_threshold: float = 0.25
    max_new_tokens: int = 512
    image_max_side: int = 1024
    load_in_4bit: bool = False


CONFIGS = {
    "grounding_dino": Config(
        "grounding_dino", "dino", "IDEA-Research/grounding-dino-tiny",
        "butterfly. moth.", candidate_threshold=0.30, threshold=0.30,
    ),
    "owlv2": Config(
        "owlv2", "owl", "google/owlv2-base-patch16-ensemble",
        ("a butterfly", "a moth"), candidate_threshold=0.10, threshold=0.10,
    ),
    "qwen": Config(
        "qwen", "qwen", "Qwen/Qwen3-VL-2B-Instruct",
        'Detect every visible adult butterfly or moth. Give each individual '
        'one tight bounding box covering only its visible parts. Ignore '
        'other insects, caterpillars, pupae, and shadows. Return only a JSON '
        'array of objects with "bbox_2d": [x_min, y_min, x_max, y_max], '
        'using integer coordinates from 0 to 1000 relative to the full image. '
        'Return [] if no adult butterfly or moth is visible.',
    ),
}

OWL_EXPERIMENT_THRESHOLDS = (0.15, 0.20, 0.25, 0.30)
import json
import math
import re


def normalized_box(coords, width, height):
    """Convert XYXY coordinates to the contest format; ignore empty boxes."""
    x1, y1, x2, y2 = coords
    if not all(math.isfinite(value) for value in coords):
        raise ValueError("Non-finite box coordinates")
    left, right = (min(1.0, max(0.0, x / width)) for x in (x1, x2))
    top, bottom = (min(1.0, max(0.0, y / height)) for y in (y1, y2))
    if right <= left or bottom <= top:
        return None
    return dict(left=left, top=top, width=right-left, height=bottom-top)


def parse_qwen(text):
    """Parse the whole response, accepting an optional Markdown JSON fence."""
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text.strip(), re.S | re.I)
    items = json.loads(fenced.group(1) if fenced else text)
    if not isinstance(items, list):
        raise ValueError("Expected a JSON list")
    boxes = []
    for item in items:
        coords = item.get("bbox_2d") if isinstance(item, dict) else None
        if not isinstance(coords, list) or len(coords) != 4 or not all(
            type(value) in (int, float) and 0 <= value <= 1000 for value in coords
        ):
            raise ValueError("Expected four bbox_2d coordinates in [0, 1000]")
        box = normalized_box(coords, 1000, 1000)
        if box is None:
            raise ValueError("Qwen returned an empty or reversed box")
        boxes.append(box)
    return boxes


class QwenResponseError(ValueError):
    def __init__(self, message, raw_response):
        super().__init__(message)
        self.raw_response = raw_response


def choose_device():
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


def load_detector(config, device):
    import torch
    from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
    from transformers import BitsAndBytesConfig

    processor = AutoProcessor.from_pretrained(config.model_id, revision=config.revision)
    if config.backend == "qwen":
        from transformers import Qwen3VLForConditionalGeneration
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        options = {"dtype": dtype}
        if config.load_in_4bit:
            if not device.startswith("cuda"):
                raise ValueError("This 4-bit configuration requires a CUDA device")
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
            )
            options["device_map"] = {"": device}
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            config.model_id, revision=config.revision, **options
        )
        if not config.load_in_4bit:
            model.to(device)
    else:
        model = AutoModelForZeroShotObjectDetection.from_pretrained(
            config.model_id, revision=config.revision
        ).to(device)
    return processor, model.eval()


def predict_one(detector, config, image, device):
    """Return candidates (each with a box and optional score) and model text."""
    import torch
    from PIL import Image

    processor, model = detector
    with torch.inference_mode():
        if config.backend == "qwen":
            resized = image.copy()
            resized.thumbnail(
                (config.image_max_side, config.image_max_side), Image.Resampling.LANCZOS
            )
            messages = [{"role": "user", "content": [
                {"type": "image", "image": resized},
                {"type": "text", "text": config.prompt},
            ]}]
            chat_text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(
                text=[chat_text], images=[resized], return_tensors="pt"
            ).to(device)
            ids = model.generate(
                **inputs, max_new_tokens=config.max_new_tokens, do_sample=False
            )
            text = processor.batch_decode(
                ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )[0]
            try:
                boxes = parse_qwen(text)
            except ValueError as error:
                raise QwenResponseError(str(error), text) from error
            return [{"box": box, "score": None} for box in boxes], text

        size = [(image.height, image.width)]
        query = config.prompt if config.backend == "dino" else [list(config.prompt)]
        inputs = processor(images=image, text=query, return_tensors="pt").to(device)
        outputs = model(**inputs)
        if config.backend == "dino":
            result = processor.post_process_grounded_object_detection(
                outputs, input_ids=inputs.input_ids, target_sizes=size,
                threshold=config.candidate_threshold, text_threshold=config.text_threshold,
            )[0]
        else:
            result = processor.image_processor.post_process_object_detection(
                outputs, target_sizes=size, threshold=config.candidate_threshold
            )[0]

    candidates = []
    for coords, score in zip(result["boxes"].tolist(), result["scores"].tolist()):
        box = normalized_box(coords, image.width, image.height)
        if box is not None:
            candidates.append({"box": box, "score": score})
    return candidates, None


def select_boxes(candidates, threshold):
    return [
        candidate["box"] for candidate in candidates
        if candidate["score"] is None or candidate["score"] >= threshold
    ]
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching


def iou_matrix(reference, predicted):
    def corners(boxes):
        xywh = np.asarray([
            [box["left"], box["top"], box["width"], box["height"]] for box in boxes
        ], dtype=float).reshape(-1, 4)
        return xywh[:, :2], xywh[:, :2] + xywh[:, 2:], np.prod(xywh[:, 2:], axis=1)

    ref_min, ref_max, ref_area = corners(reference)
    pred_min, pred_max, pred_area = corners(predicted)
    top_left = np.maximum(ref_min[:, None], pred_min[None, :])
    bottom_right = np.minimum(ref_max[:, None], pred_max[None, :])
    intersection = np.maximum(bottom_right - top_left, 0).prod(axis=2)
    union = ref_area[:, None] + pred_area[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


def summarize(rows, suffix):
    tp, fp, fn = (sum(row[f"{key}_{suffix}"] for row in rows) for key in ("tp", "fp", "fn"))
    return {
        f"tp_{suffix}": tp,
        f"fp_{suffix}": fp,
        f"fn_{suffix}": fn,
        f"precision_{suffix}": tp / (tp + fp) if tp + fp else 0.0,
        f"recall_{suffix}": tp / (tp + fn) if tp + fn else 0.0,
        f"f1_{suffix}": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    }


def evaluate(photo_ids, golden, predictions):
    rows = []
    for photo_id in photo_ids:
        reference = golden[photo_id]["boxes"]
        record = predictions[photo_id]
        error = record.get("error")
        predicted = [] if error else record["boxes"]
        ious = iou_matrix(reference, predicted)
        row = {"photo_id": photo_id, "is_negative": not reference, "error": error}
        for threshold, suffix in ((0.5, "50"), (0.75, "75")):
            tp = 0
            if ious.size:
                matches = maximum_bipartite_matching(csr_matrix(ious >= threshold))
                tp = int(np.count_nonzero(matches >= 0))
            row.update({
                f"tp_{suffix}": tp, f"fp_{suffix}": len(predicted) - tp,
                f"fn_{suffix}": len(reference) - tp,
            })
        rows.append(row)

    summary = {"n_photos": len(rows), "technical_errors": sum(bool(row["error"]) for row in rows)}
    for suffix in ("50", "75"):
        summary.update(summarize(rows, suffix))
    negatives = [row for row in rows if row["is_negative"]]
    summary.update(
        n_negative_photos=len(negatives),
        negative_photos_with_fp=sum(row["fp_50"] > 0 for row in negatives),
        fp_on_negative_photos=sum(row["fp_50"] for row in negatives),
    )
    return summary, rows
import csv
import gc
import hashlib
import json
import math
from dataclasses import asdict, replace
from io import BytesIO
from pathlib import Path
from time import perf_counter
from urllib.request import urlopen

from PIL import Image, ImageOps


def read_manifest(path):
    """Read the team JSON array (or JSONL) containing photo_id, url and group."""
    path = Path(path)
    with path.open(encoding="utf-8-sig") as file:
        if path.suffix.lower() == ".jsonl":
            records = [json.loads(line) for line in file if line.strip()]
        else:
            records = json.load(file)
    if not isinstance(records, list) or not records:
        raise ValueError(f"{path}: ожидается непустой список фотографий")

    rows = {}
    for index, record in enumerate(records, 1):
        if not isinstance(record, dict):
            raise ValueError(f"{path}: запись {index} должна быть объектом")
        photo_id = record.get("photo_id")
        url = record.get("url")
        group = record.get("group")
        if type(photo_id) is not int or photo_id <= 0 or photo_id in rows:
            raise ValueError(f"{path}: неверный или повторный photo_id {photo_id!r}")
        if not isinstance(url, str) or not url.strip().startswith(("https://", "http://")):
            raise ValueError(f"{path}: для photo_id {photo_id} нужна ссылка в url")
        if not isinstance(group, str) or not group.strip():
            raise ValueError(f"{path}: для photo_id {photo_id} нужна непустая group")
        rows[photo_id] = {"image": url.strip(), "group": group.strip()}
    return rows


def read_contest_manifest(path):
    path = Path(path)
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not {"photo_id", "image"}.issubset(reader.fieldnames or []):
            raise ValueError(f"{path}: нужны колонки photo_id и image")
        rows = {}
        for row in reader:
            photo_id = int(row["photo_id"])
            source = (row["image"] or "").strip()
            if not source or photo_id in rows:
                raise ValueError(f"{path}: пустой путь или повтор ID {photo_id}")
            if not source.startswith(("https://", "http://")):
                source = str((path.parent / source).resolve())
            rows[photo_id] = {"image": source, "group": (row.get("group") or "").strip()}
    if not rows:
        raise ValueError(f"Пустой манифест: {path}")
    return rows


def open_image(source):
    if source.startswith(("https://", "http://")):
        with urlopen(source, timeout=30) as response:
            source = BytesIO(response.read())
    with Image.open(source) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def read_jsonl(path):
    rows = {}
    with Path(path).open(encoding="utf-8-sig") as file:
        for line in file:
            if not line.strip():
                continue
            row = json.loads(line)
            record_id = row.get("photo_id")
            if type(record_id) is not int or record_id in rows:
                raise ValueError(f"{path}: неверный или повторный photo_id {record_id!r}")
            rows[record_id] = row
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    temp.replace(path)


def read_settings(raw_path):
    path = Path(raw_path).with_suffix(".settings.json")
    return json.loads(path.read_text(encoding="utf-8"))


def run_model(config, manifest, raw_path, device):
    """Сохраняет кандидатов после каждого фото. При повторе обрабатывает сбои."""
    import torch
    import transformers
    from huggingface_hub import model_info

    raw_path = Path(raw_path)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    settings_path = raw_path.with_suffix(".settings.json")
    previous = read_settings(raw_path) if settings_path.exists() else None
    if raw_path.exists() and previous is None:
        raise ValueError(f"Для {raw_path} отсутствует файл настроек")

    if config.revision == "main":
        revision = previous["config"]["revision"] if previous else model_info(config.model_id).sha
        config = replace(config, revision=revision)

    settings = {
        "config": asdict(config),
        "device": device,
        "gpu": torch.cuda.get_device_name(device) if device.startswith("cuda") else None,
        "torch": str(torch.__version__),
        "transformers": transformers.__version__,
        "manifest_hash": hashlib.sha256(
            json.dumps(manifest, sort_keys=True).encode("utf-8")
        ).hexdigest(),
    }
    settings_text = json.dumps(settings, ensure_ascii=False, indent=2)
    if previous is not None and previous != json.loads(settings_text):
        raise ValueError("Настройки или набор фото изменились. Укажи новый путь для результатов.")
    if previous is None:
        settings_path.write_text(settings_text, encoding="utf-8")

    results = read_jsonl(raw_path) if raw_path.exists() else {}
    pending = [pid for pid in manifest if pid not in results or results[pid]["error"]]
    if not pending:
        return results

    detector = load_detector(config, device)
    try:
        for index, photo_id in enumerate(pending, 1):
            previous_attempts = results.get(photo_id, {}).get("attempt", 0)
            record = {"photo_id": photo_id, "attempt": previous_attempts + 1}
            try:
                image = open_image(manifest[photo_id]["image"])
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                started = perf_counter()
                candidates, text = predict_one(detector, config, image, device)
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                record.update(candidates=candidates, raw_response=text,
                              seconds=perf_counter() - started, error=None)
            except Exception as error:
                record.update(candidates=None, seconds=None,
                              raw_response=getattr(error, "raw_response", None),
                              error=f"{type(error).__name__}: {error}")

            if record["error"] or previous_attempts:
                log_path = raw_path.with_suffix(".errors.jsonl")
                with log_path.open("a", encoding="utf-8") as log:
                    log.write(json.dumps(record, ensure_ascii=False) + "\n")

            results[photo_id] = record
            write_jsonl(raw_path, results.values())
            # if index == 1 or index % 10 == 0 or index == len(pending):
            print(f"{config.name}: {index}/{len(pending)}", flush=True)
    finally:
        del detector
        gc.collect()
        if device.startswith("cuda"):
            torch.cuda.empty_cache()
    return results


def export_predictions(raw, manifest, threshold, path):
    if set(raw) != set(manifest):
        raise ValueError("ID результатов и манифеста должны совпадать")
    rows = {}
    for photo_id in manifest:
        record = raw[photo_id]
        boxes = None if record["error"] else select_boxes(record["candidates"], threshold)
        rows[photo_id] = {"photo_id": photo_id, "boxes": boxes, "error": record["error"]}
    write_jsonl(path, rows.values())
    return rows


def check_boxes(boxes):
    if not isinstance(boxes, list):
        raise ValueError("boxes должен быть списком")
    for box in boxes:
        x, y, w, h = (box[key] for key in ("left", "top", "width", "height"))
        if not all(type(v) in (float, int) and math.isfinite(v) for v in (x, y, w, h)):
            raise ValueError(f"Неверные координаты: {box}")
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                and x+w <= 1+1e-9 and y+h <= 1+1e-9):
            raise ValueError(f"Рамка выходит за границы или имеет нулевой размер: {box}")


def validate_submission(path, manifest):
    rows = read_jsonl(path)
    if len(manifest) != 400 or set(rows) != set(manifest):
        raise ValueError("В посылке должны быть ровно 400 выданных photo_id")
    if Path(path).stat().st_size > 5 * 1024**2:
        raise ValueError("Посылка больше 5 МиБ")
    for photo_id, row in rows.items():
        if row.get("error") is not None:
            raise ValueError(f"Не обработана фотография {photo_id}: {row['error']}")
        check_boxes(row["boxes"])


ЗДЕСЬ ЗАГРУЗКА ДАТАСЕТА, ПРОВЕРИТЬ У СЕБЯ ПУТИ: ОТКУДА И КУДА ГРУЗИШЬ

In [11]:
from pathlib import Path

inputs = Path("/kaggle/input")
val_files = sorted(inputs.rglob("val_dataset*.json"))
golden_files = sorted(inputs.rglob("golden*.jsonl"))

if len(val_files) != 1 or len(golden_files) != 1:
    raise ValueError(f"Ожидаю по одному файлу. Найдено: {val_files=}, {golden_files=}")

team = read_manifest(val_files[0])
golden = read_jsonl(golden_files[0])

assert set(team) == set(golden), "photo_id в датасете и эталоне различаются"

RESULTS = Path("/kaggle/working/results")
print(f"Загружено {len(team)} фото и {len(golden)} записей эталона")

Загружено 400 фото и 400 записей эталона


In [ ]:
from dataclasses import replace
from time import perf_counter

from huggingface_hub import model_info

config = CONFIGS["qwen"]
device = choose_device()

print("1. Открываю фото", flush=True)
image = open_image(team[760598669]["image"])

print("2. Запрашиваю ревизию на Hugging Face", flush=True)
revision = model_info(config.model_id, timeout=20).sha
config = replace(config, revision=revision)

print(f"3. Загружаю веса на {device}", flush=True)
detector = load_detector(config, device)

print("4. Считаю предсказание", flush=True)
start = perf_counter()
candidates, _ = predict_one(detector, config, image, device)
print(f"Готово за {perf_counter() - start:.1f} с, рамок: {len(candidates)}")

In [ ]:
device = choose_device()
PREVIEW_MODEL = "qwen"
PREVIEW_PHOTO_ID = 760598669
config = CONFIGS[PREVIEW_MODEL]

preview_path = RESULTS / "preview" / f"{PREVIEW_MODEL}_{PREVIEW_PHOTO_ID}.jsonl"
preview = run_model(config, {PREVIEW_PHOTO_ID: team[PREVIEW_PHOTO_ID]}, preview_path, device)
record = preview[PREVIEW_PHOTO_ID]
if record["error"]:
    print("Сырой ответ:", record.get("raw_response"))
    raise RuntimeError(f"{record['error']} | подробности в {preview_path}")

image = open_image(team[PREVIEW_PHOTO_ID]["image"])
predicted_boxes = select_boxes(record["candidates"], config.threshold)
check_boxes(predicted_boxes)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, color, rectangles in (
    (axes[0], "Эталон", "lime", golden[PREVIEW_PHOTO_ID]["boxes"]),
    (axes[1], PREVIEW_MODEL, "orange", predicted_boxes),
):
    ax.imshow(image)
    for box in rectangles:
        ax.add_patch(Rectangle(
            (box["left"] * image.width, box["top"] * image.height),
            box["width"] * image.width, box["height"] * image.height,
            fill=False, edgecolor=color, linewidth=2,
        ))
    ax.set_title(f"{title} · рамок: {len(rectangles)}")
    ax.axis("off")
plt.show()

**На данных проекта.** После проверки запускаем три модели на всём наборе. Здесь сохраняются сырые ответы с уверенностью, временем и возможными сбоями.

In [ ]:
name = 'qwen'
config = CONFIGS[name]
raw = run_model(config, team, RESULTS / "raw" / f"{name}.jsonl", device)
assert set(raw) == set(team), f"У {name} неполный набор ID"
failures = sum(bool(row["error"]) for row in raw.values())
print(f"{name} | фото: {len(raw)} | сбоев: {failures}")

выгрузка в файл:

In [ ]:
baseline_runs = {name: (name, cfg.threshold) for name, cfg in [('qwen', CONFIGS['qwen'])]}
predictions_by_run = {}
timing_by_model = {}
baseline_rows = []

for name, (_, threshold) in baseline_runs.items():
    raw = raw_by_model[name]
    predictions_by_run[name] = export_predictions(
        raw, team, threshold, RESULTS / "predictions" / f"{name}.jsonl"
    )
    times = [row["seconds"] for row in raw.values() if row["seconds"] is not None]
    timing_by_model[name] = {
        "inference_seconds": sum(times),
        "ms_per_photo": 1000 * sum(times) / len(times) if times else None,
    }
    score, _ = evaluate(team, golden, predictions_by_run[name])
    baseline_rows.append({"run": name, "threshold": threshold, **score, **timing_by_model[name]})

sort_columns = ["f1_50", "ms_per_photo", "run"]
sort_directions = [False, True, True]
columns = [
    "run", "threshold", "f1_50", "f1_75", "precision_50", "recall_50",
    "tp_50", "fp_50", "fn_50", "negative_photos_with_fp",
    "fp_on_negative_photos", "technical_errors", "ms_per_photo",
]
baseline = pd.DataFrame(baseline_rows).sort_values(sort_columns, ascending=sort_directions)
baseline.to_csv(RESULTS / "qwen_board.csv", index=False)
display(baseline[columns])